# 1. Environment and Dependencies

In [5]:
# !pip install nibabel fastai kornia tqdm ipywidgets
import os
import glob
import gc
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import nibabel as nib
import cv2
from tqdm.notebook import tqdm
from ipywidgets import interact, IntSlider, fixed
from PIL import Image

# FastAI Medical Imaging
from fastai.basics import *
from fastai.vision.all import *
from fastai.data.transforms import *

# Constant Configuration
DATASOURCE = "/media/volume/Data/Projects/LITS/data/LiverTumor"
MASK_SOURCE = os.path.join(DATASOURCE, "masks/segmentations")
VOLUME_SOURCE = os.path.join(DATASOURCE, "volumes")

# 2. Data Wrangling & NIfTI Utilities
Simplified file collection logic, more robust. Instead of nested loops, we’ll map the Volume IDs directly to their corresponding masks.

In [2]:
def read_nii(filepath):
    '''Reads .nii file, rotates for standard viewing, and returns pixel array'''
    ct_scan = nib.load(filepath)
    array = ct_scan.get_fdata()
    return np.rot90(np.array(array))

# Create structured DataFrame
ct_files = sorted(glob.glob(f"{VOLUME_SOURCE}/volume-*.nii"))
df_files = pd.DataFrame(ct_files, columns=['ct_path'])
df_files['filename'] = df_files['ct_path'].apply(lambda x: os.path.basename(x))

# Map Volume ID to Mask Path
def get_mask_path(ct_filename):
    file_id = ct_filename.split('-')[-1]
    return os.path.join(MASK_SOURCE, f"segmentation-{file_id}")

df_files['mask_path'] = df_files['filename'].apply(get_mask_path)

print(f"Total CT Scans found: {len(df_files)}")

Total CT Scans found: 114


# 3. Medical Imaging Preprocessing
In medical imaging, raw Hounsfield Units (HU) are often too wide for neural networks. We use Windowing to focus on the density of specific organs (Liver/Abdomen).

In [3]:
# Custom Windowing Definitions
dicom_windows = types.SimpleNamespace(
    liver=(150, 30),
    abdomen_soft=(400, 50),
    custom=(200, 60)
)

@patch
def windowed(self:Tensor, w, l):
    """Applies Hounsfield Unit windowing to a Tensor"""
    px = self.clone()
    px_min, px_max = l - w//2, l + w//2
    px[px < px_min] = px_min
    px[px > px_max] = px_max
    return (px - px_min) / (px_max - px_min)

# 4. Advanced Visualization
Merged plotting functions into a single, cleaner version. This cell also includes an Interactive Slider so you can scroll through the 3D volume slices without re-running the cell.

In [7]:
def display_slices(ct_array, mask_array, slice_idx):
    ct_slice = ct_array[..., slice_idx]
    mask_slice = mask_array[..., slice_idx]
    
    fig, ax = plt.subplots(1, 3, figsize=(20, 7))
    
    # Raw CT
    ax[0].imshow(ct_slice, cmap='bone')
    ax[0].set_title(f'Raw CT (Slice {slice_idx})')
    
    # Windowed for Liver
    windowed_slice = tensor(ct_slice.astype(np.float32)).windowed(*dicom_windows.liver)
    ax[1].imshow(windowed_slice, cmap='bone')
    ax[1].set_title('Windowed (Liver)')
    
    # Overlay
    ax[2].imshow(windowed_slice, cmap='bone')
    ax[2].imshow(mask_slice, alpha=0.4, cmap='nipy_spectral')
    ax[2].set_title('Liver Mask Overlay')
    
    for a in ax: a.axis('off')
    plt.show()

# Load Sample for Interaction
sample_idx = 0
sample_ct = read_nii(df_files.iloc[sample_idx]['ct_path'])
sample_mask = read_nii(df_files.iloc[sample_idx]['mask_path'])

# Create Interactive Widget
interact(display_slices, 
         ct_array=fixed(sample_ct), 
         mask_array=fixed(sample_mask), 
         slice_idx=IntSlider(min=0, max=sample_ct.shape[-1]-1, step=1, value=50))

interactive(children=(IntSlider(value=50, description='slice_idx', max=122), Output()), _dom_classes=('widget-…

<function __main__.display_slices(ct_array, mask_array, slice_idx)>

# 5. Memory Management & Class Distribution
Before training, it’s important to see how many pixels belong to each class (0: Background, 1: Liver, 2: Tumor).

In [8]:
def check_mask_stats(mask_array):
    unique, counts = np.unique(mask_array, return_counts=True)
    stats = pd.DataFrame(np.array((unique, counts)).T, columns=['Class', 'Count'])
    print(stats)
    
check_mask_stats(sample_mask)

# Cleanup to free GPU/RAM
gc.collect()

   Class       Count
0    0.0  31592739.0
1    1.0    643779.0
2    2.0      7194.0


91782